In [0]:
# List tables in the delta share "accuweather"
tables = [row.tableName for row in spark.sql('SHOW TABLES IN samples.accuweather').collect()]

import pandas as pd

table_info = []
for table in tables:
    df = spark.sql(f'SELECT * FROM samples.accuweather.{table} LIMIT 1')
    table_info.append({"Table": table, "Columns": ", ".join(df.columns)})

display(pd.DataFrame(table_info))

In [0]:
# Compare average temperature from historical and forecast daily calendar imperial tables

historical_df = spark.sql("""
    SELECT city_name, date, temperature_avg AS avg_temp, 'historical' AS source
    FROM samples.accuweather.historical_daily_calendar_imperial
""")

forecast_df = spark.sql("""
    SELECT city_name, date, temperature_avg AS avg_temp, 'forecast' AS source
    FROM samples.accuweather.forecast_daily_calendar_imperial
""")

combined_df = historical_df.unionByName(forecast_df)

top_cities_df = combined_df.groupBy("city_name").agg({"avg_temp": "avg"}).withColumnRenamed("avg(avg_temp)", "avg_temp") \
    .orderBy("avg_temp", ascending=False).limit(5)

display(top_cities_df)

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
# Compare average temperature from historical and forecast daily calendar imperial tables

historical_df = spark.sql("""
    SELECT city_name, date, temperature_avg AS avg_temp, 'historical' AS source
    FROM samples.accuweather.historical_daily_calendar_imperial
""")

forecast_df = spark.sql("""
    SELECT city_name, date, temperature_avg AS avg_temp, 'forecast' AS source
    FROM samples.accuweather.forecast_daily_calendar_imperial
""")

combined_df = historical_df.unionByName(forecast_df)

from pyspark.sql.functions import col

combined_df = combined_df.withColumn("avg_temp_celsius", (col("avg_temp") - 32) * 5 / 9)

daily_df = combined_df.select("city_name", "date", "avg_temp", "avg_temp_celsius", "source").orderBy("city_name", "date")

display(daily_df)

Databricks visualization. Run in Databricks to view.

In [0]:
# Create a Delta table from the daily_df DataFrame
daily_df.write.format("delta").mode("overwrite").saveAsTable("accuweather_notebook_daily")